In [ ]:
### Imports

import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
### Create DataFrame

# I used an excel table, but csv works too, if you prefer
df = pd.read_excel('PATH_TO_FILE')

In [ ]:
# Calculate the porcentage of missiong data
# You should repeat this process for all columns that contain temperature data
# I had daily mean, max and minimum temperature columns, so i did this 3 times
# World Meteorological Organization recomends no more than 20% missing data
# However you shold consider the missing data pattern
# If you have data missing for long consecutive periods, you shold consider another method for filling in missing data because the one showed below is fragile in those scenarios
print('Percent missing data for Min Temp:')
pct_missing = df['tempMin'].isna().mean() * 100
print(round(pct_missing,2))
print('Percent missing data for Mean Temp:')
pct_missing = df['tempMed'].isna().mean() * 100
print(round(pct_missing,2))
print('Percent missing data for Max Temp:')
pct_missing = df['tempMax'].isna().mean() * 100
print(round(pct_missing,2))
print('Percent missing data for humidity:')
pct_missing = df['HumidadeMed'].isna().mean() * 100
print(round(pct_missing,2))

In [ ]:
# Filter the data by year to calculate your T95
# We used a 30 year reference period (1981-2020)
startT95 = pd.to_datetime('REFERENCE_PERIOD_START')
endT95 = pd.to_datetime('REFERENCE_PERIOD_END')
# Define T95 based on the filter
dfT95 = df[(df['date'] >= startT95) & (df['date'] <= endT95)]
# Calculate T95
T95 = dfT95['tempMean'].quantile(0.95)

In [ ]:
# Make sure your data is ordered
startDate = df['date'].min()
endDate = df['date'].max()
complete_index = pd.date_range(start=startDate, end=endDate)
df = df.set_index('date').reindex(complete_index).reset_index()
df['tempMean'] = df['tempMean'].fillna(method='ffill').fillna(method='bfill')
df['tempMax'] = df['tempMax'].fillna(method='ffill').fillna(method='bfill')
df['tempMin'] = df['tempMin'].fillna(method='ffill').fillna(method='bfill')
df['Humidity'] = df['Humidity'].fillna(method='ffill').fillna(method='bfill')
# Calculate the average temperature for the 3 day period and the 30 day period
df['TDP'] = df['tempMean'].shift(-2).rolling(window=3, min_periods=1).mean()
df['30DP'] = df['tempMean'].rolling(window=30, min_periods=1).mean()
# Keep in mind you should not use the 30 first days of your dataframe due to the calculation of the 30 day period
# Calculate EHIaccl
df['EHIaccl'] = df['TDP'] - df['30DP']
# Calculate EHIsigg
df['EHIsigg'] = df['TDP'] - T95
# Calculate EHF
df['EHF'] =  np.where(df['EHIaccl'] > 1, df['EHIsigg'] * df['EHIaccl'], df['EHIsigg'] * 1)
# Define the Heat Wave days
df['isHW'] = False
for i in range(len(df)):
    if df.loc[i, 'EHF'] > 0:
        df.loc[i:i+2, 'isHW'] = True

In [ ]:
# Caculate EHF85
positiveEHFs = df[df['EHF'] > 0]
EHF85 = positiveEHFs['EHF'].quantile(0.85)
EHF85_3x = EHF85 * 3

In [ ]:
# Define the conditions and name possible heat wave intensities based on the EHF value
conditions = [
    (df['EHF'] <= 0),
    (df['EHF'] > 0) & (df['EHF'] < EHF85),
    (df['EHF'] >= EHF85) & (df['EHF'] < EHF85_3x),
    (df['EHF'] >= EHF85_3x)
]
intensities = ['Not HW','Low-Intensity','Severe', 'Extreme']
# Define the heat wave days's intensities
df['HWDay_Intensity'] = np.select(conditions, intensities)

# Calculate daily thermal range
df['thermalRange'] = df['tempMax'] - df['tempMin']

# Create a 'year' column to facilitate future calculations
df['year'] = df['index'].dt.year

In [ ]:
# Calculate the number of heat wave days per year
HWDays = df.groupby('year')['isHW'].sum()
print(HWDays)

In [ ]:
# Define a function to count the number of heat waves per year
def count_HW_periods(df):
    df['group'] = (df['isHW'] != df['isHW'].shift()).cumsum()
    HW_periods = df[df['isHW']].groupby(['year', 'group']).size().reset_index(name='count')
    HW_periods = HW_periods[HW_periods['count'] >= 3]
    result = HW_periods.groupby('year').size().reset_index(name='num_HW_periods')
    return result

# Count the number of heat waves per year
result = count_HW_periods(df)
print(result)

In [ ]:
# Define a function to calculate the mean heat wave duration per year
def calculate_average_duration(df):
    # Identify the groups of heat wave periods
    df['group'] = (df['isHW'] != df['isHW'].shift()).cumsum()

    # Filter the heat wave periods
    HW_periods = df[df['isHW']].groupby(['year', 'group']).size().reset_index(name='count')

    # Filter only the periods with at least 3 days
    HW_periods = HW_periods[HW_periods['count'] >= 3]

    # Calculate the average heat wave duration per year
    average_duration = HW_periods.groupby('year')['count'].mean().reset_index(name='average_duration')

    return average_duration

# Print the average heat wave duration per year
average_duration = calculate_average_duration(df)
print(average_duration)

In [ ]:
# Create new column for the mean thermal range (MTR) and assing initial value
df['MTR'] = 0
df['MedTR'] = 0

# Identify periods with consecutive true values for isHW
in_period = False
start_idx = 0

for i in range(len(df)):
    if df.loc[i, 'isHW']:
        if not in_period:
            in_period = True
            start_idx = i
    else:
        if in_period:
            in_period = False
            # Calculate the mean and median for the daily thermal range values in each heat wave
            end_idx = i
            mean_thermalRange = df.loc[start_idx:end_idx-1, 'thermalRange'].mean()
            median_thermalRange = df.loc[start_idx:end_idx-1, 'thermalRange'].median()
            # Fill in the MTR column with the calculated mean
            df.loc[start_idx:end_idx-1, 'MTR'] = mean_thermalRange
            df.loc[start_idx:end_idx-1, 'MedTR'] = median_thermalRange

# Treat the last period in case it is the last dataframe entry
if in_period:
    mean_thermalRange = df.loc[start_idx:, 'thermalRange'].mean()
    df.loc[start_idx:, 'MTR'] = mean_thermalRange

In [ ]:
# Create a function to calculate number of heatwaves per month across the dataframe
# This is intended to find out which are the monsths that heat waves happen most commonly
def count_heatWaves_by_month(df):
    # Identify groups
    df['group'] = (df['isHW'] != df['isHW'].shift()).cumsum()

    # Filter the heat waves
    Heat_Waves = df[df['isHW']].groupby(['group']).agg(
        start_date=('index', 'first'),
        end_date=('index', 'last')
    ).reset_index()

    # Extract the month of the start of the heat wave
    Heat_Waves['month'] = Heat_Waves['start_date'].dt.month

    # Count the number of heat waves per month across the years
    HW_by_month = Heat_Waves.groupby(['month']).size().reset_index(name='num_periods')

    return HW_by_month

# Executar a função
HW_by_month = count_heatWaves_by_month(df)
print(HW_by_month)

In [ ]:
# Create e homogenous column to define the intensity of each heat wave instead of each heat wave day

# Auxilery function to determine the most frequent value in the HWDay_Intensity
def definir_hw_intensity(grupo):
    # Check the most frequent value in HWDay_Intensity
    most_frequent_value = grupo['HWDay_Intensity'].mode()[0]

    # Apply rules to define the HW Intensity
    if most_frequent_value in ['Not HW', 'Low-Intensity']:
        return 'Low Intensity'
    elif most_frequent_value == 'Severe':
        return 'Severe'
    elif most_frequent_value == 'Extreme':
        return 'Extreme'

# Create column with 'Not HW' for all days
df['HW_Intensity'] = 'Not HW'

# Identify consecutive True values in isHW
df['bloco'] = (df['isHW'] != df['isHW'].shift()).cumsum()

# Filter only the blocks where isHW is True (which are the heat wave periods)
blocos_hw = df[df['isHW'] == True].groupby('bloco')

# Apply the function in each block to define the value for HW_Intensity
for nome_bloco, grupo in blocos_hw:
    valor_intensidade = definir_hw_intensity(grupo)
    df.loc[df['bloco'] == nome_bloco, 'HW_Intensity'] = valor_intensidade

# Remove the temporary column
df.drop(columns=['bloco'], inplace=True)

In [ ]:
# Criar a nova coluna 'days_since_HW' com valores padrão 0
df['days_since_HW'] = 0

# Variável para contar os dias desde o último True em 'isHW'
dias_desde_ultima_hw = 0

# Variável para armazenar o valor dos dias desde a última onda de calor para ser replicado nos blocos de True
valor_atual_hw = 0

# Iterar sobre o DataFrame linha por linha
for i in range(len(df)):
    if df['isHW'].iloc[i] == True:
        # Se for o primeiro dia da nova onda de calor, definir o valor do bloco como o contador atual
        if df['isHW'].iloc[i-1] == False or i == 0:
            valor_atual_hw = dias_desde_ultima_hw

        # Atribuir o mesmo valor para todos os dias de onda de calor
        df['days_since_HW'].iloc[i] = valor_atual_hw

        # Reseta o contador após o início da onda de calor
        dias_desde_ultima_hw = 0
    else:
        # Incrementar o contador de dias entre as ondas de calor
        dias_desde_ultima_hw += 1
        # Atribuir 0 aos dias entre as ondas de calor
        df['days_since_HW'].iloc[i] = 0

In [ ]:
# Create new column for the mean humidity and assing initial value
df['Mean_HW_Humidity'] = 0

# Identify periods with consecutive true values for isHW
in_period = False
start_idx = 0

for i in range(len(df)):
    if df.loc[i, 'isHW']:
        if not in_period:
            in_period = True
            start_idx = i
    else:
        if in_period:
            in_period = False
            # Calculate the mean humidity in each heat wave
            end_idx = i
            mean_humidity = df.loc[start_idx:end_idx-1, 'HumidadeMed'].mean()
            # Fill in the MeanHumidity column with the calculated mean
            df.loc[start_idx:end_idx-1, 'Mean_HW_Humidity'] = mean_humidity

# Treat the last period in case it is the last dataframe entry
if in_period:
    mean_humidity = df.loc[start_idx:, 'HumidadeMed'].mean()
    df.loc[start_idx:, 'Mean_HW_Humidity'] = mean_humidity

In [ ]:
# Create new column 'HW_duration' with placeholder value
df['HW_duration'] = 0

# Identify blocks of consecutive True valuers in isHW
df['bloco'] = (df['isHW'] != df['isHW'].shift()).cumsum()

# Filter only the blocks where isHW is True (the heat wave periods)
blocos_hw = df[df['isHW'] == True].groupby('bloco')

# calculate the duration and atribute it to the HW_duration for each heat wave block
for nome_bloco, grupo in blocos_hw:
    duracao = len(grupo)  # Calculate HW duration
    df.loc[df['bloco'] == nome_bloco, 'HW_duration'] = duracao

# Remove temporary column
df.drop(columns=['bloco'], inplace=True)

In [ ]:
# Filter the DataFrame for the 1981 to 2010 period
df_referencia = df[(df['index'].dt.year >= 1981) & (df['index'].dt.year <= 2010)]

# Create a column for the month and day to group the data
df_referencia['mes_dia'] = df_referencia['index'].dt.strftime('%m-%d')

# Calculate the mean maximum temperature for each day during the selected period
media_tempMax = df_referencia.groupby('mes_dia')['tempMax'].mean()

# Create a reference column in the original DataFrame
df['mes_dia'] = df['index'].dt.strftime('%m-%d')

# Calculate maximum temperature anomaly
df['tempAnom'] = df['tempMax'] - df['mes_dia'].map(media_tempMax)

# Drop the temporary column
df.drop(columns=['mes_dia'], inplace=True)

# Create column for the HeatWave mean temperature anomaly with placeholder value
df['mean_temp_anom'] = 0

# Identify the heatwaves
df['bloco'] = (df['isHW'] != df['isHW'].shift()).cumsum()

# Filter the heatwave blocks
blocos_hw = df[df['isHW'] == True].groupby('bloco')

# Calculate mean temperature anomaly with the maximum temperatures for the heatwaves
for nome_bloco, grupo in blocos_hw:
    # Calculate the mean anomalies from 'tempAnom'
    media_anomalia = grupo['tempAnom'].mean()
    # Assign the calculated mean to the 'mean_temp_anom' for the heatwave days
    df.loc[df['bloco'] == nome_bloco, 'mean_temp_anom'] = media_anomalia

# Drop the temporary column
df.drop(columns=['bloco'], inplace=True)

In [ ]:
### Exporta a tabela
df.to_excel('data.xlsx', index=False)
### Download da tabela
from google.colab import files
files.download('data.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>